# ABIDES pipeline

Run the short smoke configuration first, then opt into a larger generated batch by changing the full-run cell.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from mbo_lab.abides import run_abides_export
from mbo_lab.paths import DATA, abides_paths
from mbo_lab.pipeline import (
    build_training_batch,
    load_abides_source,
    run_training_pipeline,
    save_training_batch,
)

Run `py -m uv sync --locked` in the repository once. If the pinned ABIDES interpreter is missing, run `py -m uv run python scripts/setup_abides.py` once. Generated observations and batches are written below `data/`; `.local/` is reserved for ABIDES source and environments.

In [2]:
SEED = 0
HISTORY = 256
HORIZON = 128
SMOKE_END_TIME = "10:00:00"
SMOKE_ANCHOR_COUNT = 64

In [3]:
smoke = run_training_pipeline(
    seed=SEED,
    end_time=SMOKE_END_TIME,
    history=HISTORY,
    horizon=HORIZON,
    anchor_count=SMOKE_ANCHOR_COUNT,
)
{key: value for key, value in smoke.report.items() if key != "feature_names"}

{'source': 'ABIDES RMSC04',
 'synthetic': True,
 'clock': 'simulation',
 'observations': 'C:\\Users\\Adarsh Arun\\Downloads\\BookSpace\\data\\simulated\\abides\\rmsc04-seed-0-until-100000\\observations.npz',
 'provenance': 'C:\\Users\\Adarsh Arun\\Downloads\\BookSpace\\data\\simulated\\abides\\rmsc04-seed-0-until-100000\\provenance.json',
 'observations_sha256': '422b5788bef8819fd099507232e1ddd57f76996c1633aeced044dbf7af05e4e5',
 'book_rows': 4595,
 'segments': 2,
 'feature_schema_version': 2,
 'feature_count': 86,
 'x_shape': [4595, 86],
 'y_shape': [4595],
 'history': 256,
 'horizon': 128,
 'split_row': 3063,
 'train_anchors': 2679,
 'validation_anchors': 1149,
 'selected_anchors': 64,
 'X_shape': [64, 256, 86],
 'Y_shape': [64, 128],
 'D_shape': [1497],
 'eligible_pairs': 1497,
 'target_scale': {'value': 4.552713045783882e-05,
  'pair_count': 1497,
  'zero_fraction': 0.0,
  'seed': 0},
 'time_delta_transform': 'log1p(time_delta_seconds / time_delta_tau), then train-only mean/std',
 

In [4]:
{
    "raw_observations": smoke.observations.x.shape,
    "X": smoke.X.shape,
    "Y": smoke.Y.shape,
    "D": smoke.D.shape,
    "time_delta_tau": smoke.feature_scale.time_delta_tau,
    "output": smoke.report["output"],
}

{'raw_observations': (4595, 86),
 'X': (64, 256, 86),
 'Y': (64, 128),
 'D': (1497,),
 'time_delta_tau': 0.570698283,
 'output': 'C:\\Users\\Adarsh Arun\\Downloads\\BookSpace\\data\\processed\\abides\\rmsc04-seed-0-until-100000\\smoke'}

## Full generation

Change the end time and anchor count below, then set `RUN_FULL = True`. `anchor_count` controls the number of training anchors retained in memory; use `None` only when the resulting pair count is manageable.

In [7]:
FULL_SEED = 1
FULL_END_TIME = "16:00:00"
FULL_ANCHOR_COUNT = 512
FULL_SOURCE, FULL_SMOKE_OUTPUT = abides_paths(FULL_SEED, FULL_END_TIME)
FULL_OUTPUT = FULL_SMOKE_OUTPUT.parent / "full"
RUN_FULL = True
USE_EXISTING_SOURCE = False

In [8]:
if RUN_FULL:
    full = run_training_pipeline(
        output=FULL_OUTPUT,
        seed=FULL_SEED,
        end_time=FULL_END_TIME,
        history=HISTORY,
        horizon=HORIZON,
        anchor_count=FULL_ANCHOR_COUNT,
    )
    {key: value for key, value in full.report.items() if key != "feature_names"}

If ABIDES has already produced `FULL_SOURCE`, set `USE_EXISTING_SOURCE = True` instead. This reuses the observations and skips the export step.

In [ ]:
if USE_EXISTING_SOURCE:
    full = run_training_pipeline(
        output=FULL_OUTPUT,
        observations_dir=FULL_SOURCE,
        seed=FULL_SEED,
        end_time=FULL_END_TIME,
        history=HISTORY,
        horizon=HORIZON,
        anchor_count=FULL_ANCHOR_COUNT,
        run_export=False,
    )
    {key: value for key, value in full.report.items() if key != "feature_names"}

The lower-level functions are available when an experiment needs to inspect or modify the source between export and batch construction: `run_abides_export`, `load_abides_source`, `build_training_batch`, and `save_training_batch`.